# Caso Práctico: Arquitectura Medallion Polyglot (Parte 1 - Bronze y Auditoría)


---

## PASO 0.1: Configuración de SparkSession (Hive), rutas y base de datos

In [1]:
import shutil
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("iberian_logistics_bronze")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

BASE_PATH = "/opt/datos/iberian_logistics_polyglot"
RAW_JSON_PATH = f"{BASE_PATH}/landing_json"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

if Path(BASE_PATH).exists():
    shutil.rmtree(BASE_PATH)

for p in [RAW_JSON_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    Path(p).mkdir(parents=True, exist_ok=True)

spark.sql("DROP DATABASE IF EXISTS iberian_logistics_db CASCADE")
spark.sql("CREATE DATABASE IF NOT EXISTS iberian_logistics_db")
spark.sql("USE iberian_logistics_db")

print("Entorno inicializado en:", BASE_PATH)
print("Master:", spark.sparkContext.master)
print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))

NOTE: Picked up JDK_JAVA_OPTIONS: -Duser.home=/home/spark -Dlog4j2.configurationFile=/opt/spark/conf/log4j2.properties --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/jdk.internal.loader=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=tru

Entorno inicializado en: /opt/datos/iberian_logistics_polyglot
Master: spark://spark-master:7077
Catálogo: hive


## PASO 0.2: Verificación del clúster Multi-Node y reparto por Executors


In [3]:
import socket

sc = spark.sparkContext
print(f"Nombre de la Aplicación Spark : {sc.appName}")
print(f"Paralelismo por defecto       : {sc.defaultParallelism} cores totales en Workers")
print(f"Particiones Shuffle SQL       : {spark.conf.get('spark.sql.shuffle.partitions')}")
def inspect_executor_partition(index, iterator):
    hostname = socket.gethostname()
    ip_addr = socket.gethostbyname(hostname)
    count = sum(1 for _ in iterator)
    yield (index, hostname, ip_addr, count)

rdd_test = sc.parallelize(range(1, 10001), numSlices=8)
partition_distribution = rdd_test.mapPartitionsWithIndex(inspect_executor_partition).collect()

df_executors = spark.createDataFrame(
    partition_distribution,
    ["partition_id", "worker_hostname", "worker_ip", "records_processed"]
)

df_executors.show(truncate=False)

Nombre de la Aplicación Spark : iberian_logistics_bronze
Paralelismo por defecto       : 3 cores totales en Workers
Particiones Shuffle SQL       : 200
+------------+---------------+----------+-----------------+
|partition_id|worker_hostname|worker_ip |records_processed|
+------------+---------------+----------+-----------------+
|0           |spark-worker-1 |172.18.0.3|1250             |
|1           |spark-worker-3 |172.18.0.4|1250             |
|2           |spark-worker-2 |172.18.0.5|1250             |
|3           |spark-worker-3 |172.18.0.4|1250             |
|4           |spark-worker-1 |172.18.0.3|1250             |
|5           |spark-worker-2 |172.18.0.5|1250             |
|6           |spark-worker-3 |172.18.0.4|1250             |
|7           |spark-worker-1 |172.18.0.3|1250             |
+------------+---------------+----------+-----------------+



## PASO 1: Generación de las 2 fuentes de datos sucias en JSON
Genera `raw_customers.json` (167 filas) y `raw_orders_shipments.json` (853 filas) en `/opt/datos/iberian_logistics_polyglot/landing_json`.

In [4]:
import json
import random
from datetime import datetime, timedelta
from pathlib import Path

random.seed(42)

cities = ["Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao", "Zaragoza", "Málaga"]
segments_dirty = ["VIP", " vip ", "Estándar", "ESTANDAR", "Básico", "basico", None]

customers_data = []
for i in range(1, 151):
    cust_id = f"CUST-{i:04d}"
    city = random.choice(cities)
    segment = random.choice(segments_dirty)
    age = random.randint(20, 70)
    email = f"cliente{i}@iberianlogistics.es"

    if i % 15 == 0:
        age = -10
    elif i % 23 == 0:
        age = 999
    if i % 18 == 0:
        email = "correo_invalido_sin_arroba.com"

    record = {
        "customer_id": cust_id,
        "full_name": f"Cliente {i}",
        "email": email,
        "city": city,
        "segment": segment,
        "age": int(age),
        "signup_date": (datetime(2025, 1, 1) + timedelta(days=random.randint(0, 300))).strftime("%Y-%m-%d")
    }
    customers_data.append(record)
    if i % 12 == 0:
        customers_data.append(record.copy())

for _ in range(5):
    customers_data.append({
        "customer_id": None,
        "full_name": "Usuario Fantasma",
        "email": "null_user@error.es",
        "city": "Madrid",
        "segment": "VIP",
        "age": 35,
        "signup_date": "2025-05-10"
    })

Path(f"{RAW_JSON_PATH}/raw_customers.json").write_text(
    "\n".join([json.dumps(r, ensure_ascii=False) for r in customers_data]),
    encoding="utf-8"
)

categories = ["Electrónica", "Hogar", "Moda", "Deportes", "Alimentación"]
carriers = ["ExpressIberia", "TransNorte", "EcoLogix", "RápidoSur"]
statuses = ["DELIVERED", "IN_TRANSIT", "CANCELLED", "RETURNED", "delivered "]

orders_data = []
base_date = datetime(2026, 1, 1)

for j in range(1, 801):
    order_id = f"ORD-{j:05d}"
    cust_id = f"CUST-{random.randint(1, 160):04d}"
    category = random.choice(categories)
    carrier = random.choice(carriers)
    status = random.choice(statuses)
    qty = random.randint(1, 10)
    unit_price = round(random.uniform(15.0, 450.0), 2)
    shipping_cost = round(random.uniform(3.5, 25.0), 2)
    promised_days = random.choice([1, 2, 3, 5])
    actual_days = max(1, promised_days + random.choice([-1, 0, 0, 0, 1, 2, 4]))

    if j % 20 == 0:
        qty = -3
    if j % 25 == 0:
        unit_price = -99.99
    if j % 30 == 0:
        cust_id = None
    if j % 40 == 0:
        actual_days = None

    order_rec = {
        "order_id": order_id,
        "customer_id": cust_id,
        "order_date": (base_date + timedelta(days=random.randint(0, 280))).strftime("%Y-%m-%d"),
        "product_category": category,
        "quantity": int(qty),
        "unit_price": float(unit_price),
        "shipping_cost": float(shipping_cost),
        "carrier": carrier,
        "delivery_status": status,
        "promised_delivery_days": int(promised_days),
        "actual_delivery_days": int(actual_days) if actual_days is not None else None
    }
    orders_data.append(order_rec)
    if j % 15 == 0:
        orders_data.append(order_rec.copy())

Path(f"{RAW_JSON_PATH}/raw_orders_shipments.json").write_text(
    "\n".join([json.dumps(r, ensure_ascii=False) for r in orders_data]),
    encoding="utf-8"
)

print(f"Ficheros generados en {RAW_JSON_PATH}:")
print(f" - raw_customers.json: {len(customers_data)} filas")
print(f" - raw_orders_shipments.json: {len(orders_data)} filas")

Ficheros generados en /opt/datos/iberian_logistics_polyglot/landing_json:
 - raw_customers.json: 167 filas
 - raw_orders_shipments.json: 853 filas


## EJERCICIO 1 - CAPA BRONZE (PySpark): Ingesta de JSON a tablas persistentes
Lee ambos JSON, añade `_ingestion_timestamp` y `_source_file`, reparticiona en 4 y guarda en formato `parquet` dentro del catálogo Hive (`iberian_logistics_db`).

In [5]:
# TODO 1.1: Leer raw_customers.json, añadir metadatos y reparticionar en 4
df_bronze_customers = (
    spark.read
    .json(f"{RAW_JSON_PATH}/raw_customers.json")
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .repartition(4)
)

# TODO 1.2: Guardar df_bronze_customers en Hive (formato parquet en Docker)
(
    df_bronze_customers.write
    .format("parquet")
    .mode("overwrite")
    .option("path", f"{BRONZE_PATH}/customers")
    .saveAsTable("iberian_logistics_db.bronze_customers")
)

# TODO 1.3: Leer raw_orders_shipments.json, añadir metadatos y reparticionar en 4
df_bronze_orders = (
    spark.read
    .json(f"{RAW_JSON_PATH}/raw_orders_shipments.json")
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .repartition(4)
)

# TODO 1.4: Guardar df_bronze_orders en Hive (formato parquet en Docker)
(
    df_bronze_orders.write
    .format("parquet")
    .mode("overwrite")
    .option("path", f"{BRONZE_PATH}/orders_shipments")
    .saveAsTable("iberian_logistics_db.bronze_orders_shipments")
)

# Verificación
print("Filas en bronze_customers:", spark.table("iberian_logistics_db.bronze_customers").count())
print("Filas en bronze_orders_shipments:", spark.table("iberian_logistics_db.bronze_orders_shipments").count())

Filas en bronze_customers: 167


Filas en bronze_orders_shipments: 853


## EJERCICIO 2 - AUDITORÍA DE CALIDAD EN BRONZE (PySpark)


In [6]:
df_bc = spark.table("iberian_logistics_db.bronze_customers")
df_bo = spark.table("iberian_logistics_db.bronze_orders_shipments")

# TODO 2.1: Auditoría de bronze_customers
total_bc = df_bc.count()
bc_null_id = df_bc.filter(F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == "")).count()
bc_bad_age = df_bc.filter(F.col("age").isNull() | (F.col("age") < 18) | (F.col("age") > 100)).count()
bc_bad_email = df_bc.filter(F.col("email").isNull() | (~F.col("email").contains("@"))).count()
bc_non_null = df_bc.filter(F.col("customer_id").isNotNull())
bc_dups = bc_non_null.count() - bc_non_null.select("customer_id").distinct().count()

print("=== AUDITORÍA: bronze_customers ===")
print(f"Total filas brutas             : {total_bc}")
print(f"customer_id nulos o vacíos     : {bc_null_id}")
print(f"Edades fuera de [18, 100]      : {bc_bad_age}")
print(f"Emails sin '@'                 : {bc_bad_email}")
print(f"Duplicados (sobre IDs no nulos): {bc_dups}")

# TODO 2.2: Auditoría de bronze_orders_shipments
total_bo = df_bo.count()
bo_null_oid = df_bo.filter(F.col("order_id").isNull()).count()
bo_null_cid = df_bo.filter(F.col("customer_id").isNull()).count()
bo_bad_qty = df_bo.filter(F.col("quantity").isNull() | (F.col("quantity") <= 0)).count()
bo_bad_price = df_bo.filter(F.col("unit_price").isNull() | (F.col("unit_price") <= 0)).count()
bo_null_days = df_bo.filter(F.col("actual_delivery_days").isNull()).count()
bo_non_null = df_bo.filter(F.col("order_id").isNotNull())
bo_dups = bo_non_null.count() - bo_non_null.select("order_id").distinct().count()

print("\n=== AUDITORÍA: bronze_orders_shipments ===")
print(f"Total filas brutas             : {total_bo}")
print(f"order_id nulos                 : {bo_null_oid}")
print(f"customer_id nulos              : {bo_null_cid}")
print(f"quantity <= 0                  : {bo_bad_qty}")
print(f"unit_price <= 0                : {bo_bad_price}")
print(f"actual_delivery_days nulos     : {bo_null_days}")
print(f"Duplicados (sobre IDs no nulos): {bo_dups}")

=== AUDITORÍA: bronze_customers ===
Total filas brutas             : 167
customer_id nulos o vacíos     : 5
Edades fuera de [18, 100]      : 18
Emails sin '@'                 : 12
Duplicados (sobre IDs no nulos): 12

=== AUDITORÍA: bronze_orders_shipments ===
Total filas brutas             : 853
order_id nulos                 : 0
customer_id nulos              : 52
quantity <= 0                  : 53
unit_price <= 0                : 42
actual_delivery_days nulos     : 26
Duplicados (sobre IDs no nulos): 53


## Cerrar sesión


In [7]:
spark.stop()
print("Sesión de Spark cerrada correctamente.")

Sesión de Spark cerrada correctamente.
